# ClipCap: Extended Labels cho Visual Prefix và GPT-2

Notebook này tập trung vào phần **extended labels** trong ClipCap. Mục tiêu là hiểu vì sao phải thêm các giá trị `-100` trước caption labels, cách GPT-2 dùng labels để tính causal language-model loss, và cách kiểm tra gradient vẫn truyền về Mapping Network.

Phạm vi của notebook:

- Dataset chỉ cung cấp `image_embed`, `input_ids` và `attention_mask`.
- Dùng dữ liệu giả để chạy độc lập, không cần tải pretrained model.
- Mô phỏng đúng hợp đồng tensor giữa Mapping Network và GPT-2.
- Tạo caption labels và extended labels tại tầng model khi cần tính loss.

## 1. Vị trí của extended labels trong pipeline

```text
CLIP feature [B, clip_dim]
        |
        v
TransformerMapper -> prefix_embeddings [B, P, D]
                                      |
input_ids [B, L] -> text_embeddings [B, L, D]
                                      |
                                      v
                         inputs_embeds [B, P + L, D]
                                      |
input_ids + attention_mask            |
        |                             |
        v                             v
extended_labels [B, P + L] --------> GPT-2 -> loss
```

Ký hiệu:

- `B`: batch size.
- `P`: số visual prefix token.
- `L`: chiều dài caption sau tokenize và padding.
- `D`: embedding dimension của GPT-2.
- `V`: vocabulary size của GPT-2.

In [1]:
from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import Tensor
from transformers import GPT2Config, GPT2LMHeadModel

torch.manual_seed(42)

print("PyTorch version    :", torch.__version__)
print("CUDA available    :", torch.cuda.is_available())

c:\Users\ADMIN\miniconda3\envs\zfs-caption\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version    : 2.11.0+cu128
CUDA available    : True


## 2. Ba loại vị trí trong sequence

| Loại vị trí | Attention mask | Label | Được dùng làm ngữ cảnh? | Có tính loss? |
|---|---:|---:|---:|---:|
| Visual prefix | `1` | `-100` | Có | Không |
| Caption token hợp lệ | `1` | Token ID | Có | Có |
| Padding | `0` | `-100` | Không | Không |

`attention_mask` và `labels` không có cùng nhiệm vụ:

- `attention_mask` cho biết GPT-2 được sử dụng vị trí nào làm ngữ cảnh.
- `labels` cho biết vị trí nào được chấm loss và token mục tiêu là gì.
- `-100` là `ignore_index` mà cross-entropy bỏ qua.

Visual prefix cần được GPT-2 đọc, nhưng nó không phải token ID trong vocabulary. Vì vậy prefix có attention mask bằng `1` nhưng label bằng `-100`.

## 3. Tạo batch caption giả

Batch dưới đây có hai caption đã padding đến cùng chiều dài `L = 6`. Token ID `0` được dùng làm padding trong ví dụ. Logic tạo labels không suy luận padding từ token ID; nó dùng `attention_mask` để tránh nhầm padding với token hợp lệ.

In [2]:
input_ids = torch.tensor(
    [
        [11, 12, 13, 14, 0, 0],
        [21, 22, 23, 24, 25, 0],
    ],
    dtype=torch.long,
)

attention_mask = torch.tensor(
    [
        [1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 0],
    ],
    dtype=torch.long,
)

B, L = input_ids.shape
P = 3

print("input_ids shape     :", tuple(input_ids.shape))
print("attention_mask shape:", tuple(attention_mask.shape))
print("prefix_length       :", P)
print("input_ids:\n", input_ids)
print("attention_mask:\n", attention_mask)

input_ids shape     : (2, 6)
attention_mask shape: (2, 6)
prefix_length       : 3
input_ids:
 tensor([[11, 12, 13, 14,  0,  0],
        [21, 22, 23, 24, 25,  0]])
attention_mask:
 tensor([[1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 0]])


## 4. Hàm `build_extended_labels`

Thiết kế đích là Dataset chỉ cần trả `input_ids` và `attention_mask`. Model tạo labels ngay trước khi gọi GPT-2:

1. Clone `input_ids` để không sửa dữ liệu đầu vào.
2. Đổi padding thành `-100` dựa trên `attention_mask`.
3. Tạo `P` prefix labels bằng `-100`.
4. Ghép prefix labels trước caption labels.

Hàm không hard-code batch size, caption length hoặc prefix length.

In [3]:
def build_extended_labels(
    input_ids: Tensor,
    attention_mask: Tensor,
    prefix_length: int,
) -> Tensor:
    """Tạo labels [B, P + L] cho ClipCap training."""
    if input_ids.ndim != 2:
        raise ValueError("input_ids must have shape [B, L]")

    if attention_mask.ndim != 2:
        raise ValueError("attention_mask must have shape [B, L]")

    if input_ids.shape != attention_mask.shape:
        raise ValueError(
            "input_ids and attention_mask must have the same shape"
        )

    if input_ids.dtype != torch.long:
        raise TypeError("input_ids must use torch.long dtype")

    if (
        isinstance(prefix_length, bool)
        or not isinstance(prefix_length, int)
        or prefix_length <= 0
    ):
        raise ValueError(
            "prefix_length must be a positive integer"
        )

    caption_labels = input_ids.clone()
    caption_labels.masked_fill_(attention_mask == 0, -100)

    batch_size = input_ids.size(0)
    prefix_labels = torch.full(
        (batch_size, prefix_length),
        fill_value=-100,
        dtype=input_ids.dtype,
        device=input_ids.device,
    )

    return torch.cat(
        [prefix_labels, caption_labels],
        dim=1,
    )

In [4]:
original_input_ids = input_ids.clone()

extended_labels = build_extended_labels(
    input_ids=input_ids,
    attention_mask=attention_mask,
    prefix_length=P,
)

print("extended_labels shape:", tuple(extended_labels.shape))
print(extended_labels)

assert extended_labels.shape == (B, P + L)
assert torch.all(extended_labels[:, :P] == -100)
assert torch.all(extended_labels[:, P:][attention_mask == 0] == -100)
assert torch.equal(
    extended_labels[:, P:][attention_mask == 1],
    input_ids[attention_mask == 1],
)
assert torch.equal(input_ids, original_input_ids)
assert extended_labels.dtype == input_ids.dtype
assert extended_labels.device == input_ids.device

print("PASS: shape, values, dtype, device and input immutability")

extended_labels shape: (2, 9)
tensor([[-100, -100, -100,   11,   12,   13,   14, -100, -100],
        [-100, -100, -100,   21,   22,   23,   24,   25, -100]])
PASS: shape, values, dtype, device and input immutability


## 5. Kiểm tra contract mới của Dataset

Dataset không tạo hoặc trả về `labels`. Nó chỉ cung cấp `image_embed`, `input_ids` và `attention_mask`.

Khi training hoặc validation cần loss, model dùng `input_ids` và `attention_mask` để gọi `build_extended_labels`. Khi generation, model không cần tạo labels.

In [ ]:
dataset_batch = {
    "image_embed": torch.randn(B, 512),
    "input_ids": input_ids,
    "attention_mask": attention_mask,
}

assert set(dataset_batch) == {
    "image_embed",
    "input_ids",
    "attention_mask",
}
assert "labels" not in dataset_batch

model_side_labels = build_extended_labels(
    input_ids=dataset_batch["input_ids"],
    attention_mask=dataset_batch["attention_mask"],
    prefix_length=P,
)

assert torch.equal(
    extended_labels,
    model_side_labels,
)

print("PASS: Dataset has no labels and model builds them when needed")

## 6. GPT-2 tự shift labels như thế nào?

GPT-2 là causal language model: output tại một vị trí dự đoán token kế tiếp. Khi nhận tham số `labels`, GPT-2 tự thực hiện phép dịch tương đương:

```python
shift_logits = logits[:, :-1, :]
shift_labels = labels[:, 1:]
```

Với sequence:

```text
inputs: [prefix_1, prefix_2, prefix_3, token_1, token_2, token_3]
labels: [-100,    -100,    -100,    token_1, token_2, token_3]
```

Output tại `prefix_3` dự đoán `token_1`. Vì GPT-2 đã shift nội bộ, không được shift labels thủ công trước khi truyền vào model.

In [6]:
shift_labels = extended_labels[:, 1:]

first_supervised_logit_position = (
    shift_labels[0] != -100
).nonzero(as_tuple=False)[0].item()

print("Extended labels, sample 0:", extended_labels[0].tolist())
print("Shift labels, sample 0   :", shift_labels[0].tolist())
print(
    "First supervised logit position:",
    first_supervised_logit_position,
)
print(
    "Expected last prefix position   :",
    P - 1,
)

assert first_supervised_logit_position == P - 1
print("PASS: the last visual prefix predicts the first caption token")

Extended labels, sample 0: [-100, -100, -100, 11, 12, 13, 14, -100, -100]
Shift labels, sample 0   : [-100, -100, 11, 12, 13, 14, -100, -100]
First supervised logit position: 2
Expected last prefix position   : 2
PASS: the last visual prefix predicts the first caption token


## 7. Tích hợp với GPT-2 nhỏ, không cần tải model

Để kiểm tra đúng hành vi của Hugging Face GPT-2 mà không tải pretrained weights, ta tạo một GPT-2 rất nhỏ từ `GPT2Config`. Model này chỉ dùng để kiểm tra shape, loss và gradient; nó không có khả năng sinh caption có ý nghĩa.

`prefix_embeddings` trong cell sau mô phỏng output `[B, P, D]` của Mapping Network.

In [ ]:
D = 32
V = 128

gpt2_config = GPT2Config(
    vocab_size=V,
    n_positions=P + L + 4,
    n_ctx=P + L + 4,
    n_embd=D,
    n_layer=2,
    n_head=4,
    bos_token_id=1,
    eos_token_id=2,
)

gpt2 = GPT2LMHeadModel(gpt2_config)
gpt2.loss_type = "ForCausalLM"
gpt2.eval()

prefix_embeddings = torch.randn(
    B,
    P,
    D,
    requires_grad=True,
)

text_embeddings = gpt2.get_input_embeddings()(input_ids)
inputs_embeds = torch.cat(
    [prefix_embeddings, text_embeddings],
    dim=1,
)

prefix_attention_mask = torch.ones(
    (B, P),
    dtype=attention_mask.dtype,
    device=attention_mask.device,
)
extended_attention_mask = torch.cat(
    [prefix_attention_mask, attention_mask],
    dim=1,
)

print("inputs_embeds shape           :", tuple(inputs_embeds.shape))
print("extended_attention_mask shape:", tuple(extended_attention_mask.shape))
print("extended_labels shape        :", tuple(extended_labels.shape))

assert inputs_embeds.shape == (B, P + L, D)
assert extended_attention_mask.shape == (B, P + L)
assert extended_labels.shape == (B, P + L)

inputs_embeds shape           : (2, 9, 32)
extended_attention_mask shape: (2, 9)
extended_labels shape        : (2, 9)


In [8]:
outputs = gpt2(
    inputs_embeds=inputs_embeds,
    attention_mask=extended_attention_mask,
    labels=extended_labels,
)

print("Loss        :", outputs.loss.item())
print("Logits shape:", tuple(outputs.logits.shape))

assert outputs.loss.ndim == 0
assert torch.isfinite(outputs.loss)
assert outputs.logits.shape == (B, P + L, V)

print("PASS: GPT-2 accepts extended labels and returns finite loss")

Loss        : 4.8456597328186035
Logits shape: (2, 9, 128)
PASS: GPT-2 accepts extended labels and returns finite loss


## 8. Kiểm chứng loss của GPT-2

Cell sau tính cross-entropy thủ công bằng logits đã shift và labels đã shift. Kết quả phải bằng loss mà GPT-2 trả về. Đây là cách kiểm tra quan trọng để hiểu rằng:

- GPT-2 tự shift.
- `-100` thực sự bị bỏ qua.
- Ta không cần viết hàm loss riêng cho trường hợp thông thường.

In [9]:
manual_shift_logits = outputs.logits[:, :-1, :].contiguous()
manual_shift_labels = extended_labels[:, 1:].contiguous()

manual_loss = F.cross_entropy(
    manual_shift_logits.view(-1, V),
    manual_shift_labels.view(-1),
    ignore_index=-100,
)

print("GPT-2 loss :", outputs.loss.item())
print("Manual loss:", manual_loss.item())

assert torch.allclose(
    outputs.loss,
    manual_loss,
    atol=1e-6,
)

print("PASS: manual causal LM loss matches GPT-2 loss")

GPT-2 loss : 4.8456597328186035
Manual loss: 4.8456597328186035
PASS: manual causal LM loss matches GPT-2 loss


## 9. Gradient vẫn truyền về Mapping Network

Prefix labels bằng `-100` chỉ có nghĩa là không tính loss trực tiếp tại các vị trí prefix. Prefix embeddings vẫn ảnh hưởng đến dự đoán caption, nên gradient từ caption loss vẫn truyền về Mapper.

Trong ví dụ này, `prefix_embeddings` là tensor lá mô phỏng output của Mapper. Nếu gradient của nó tồn tại và hữu hạn sau `backward()`, luồng gradient từ caption loss về prefix đã hoạt động.

In [10]:
gpt2.zero_grad(set_to_none=True)
if prefix_embeddings.grad is not None:
    prefix_embeddings.grad = None

outputs.loss.backward()

assert prefix_embeddings.grad is not None
assert torch.isfinite(prefix_embeddings.grad).all()
assert prefix_embeddings.grad.abs().sum() > 0

print("Prefix gradient shape:", tuple(prefix_embeddings.grad.shape))
print("Gradient L1 norm    :", prefix_embeddings.grad.abs().sum().item())
print("PASS: caption loss sends gradient to visual prefix")

Prefix gradient shape: (2, 3, 32)
Gradient L1 norm    : 0.11548250913619995
PASS: caption loss sends gradient to visual prefix


## 10. Freeze GPT-2 không đồng nghĩa với `torch.no_grad()`

Khi chỉ muốn train Mapping Network, có thể freeze tham số GPT-2:

```python
for parameter in gpt2.parameters():
    parameter.requires_grad = False
```

Autograd vẫn theo dõi phép tính để truyền gradient về prefix embeddings. Không nên bọc toàn bộ GPT-2 forward bằng `torch.no_grad()` khi cần train Mapper, vì nó sẽ cắt gradient về Mapper.

In [11]:
frozen_gpt2 = GPT2LMHeadModel(gpt2_config)
frozen_gpt2.loss_type = "ForCausalLM"
for parameter in frozen_gpt2.parameters():
    parameter.requires_grad = False

frozen_prefix = torch.randn(B, P, D, requires_grad=True)
frozen_text_embeddings = frozen_gpt2.get_input_embeddings()(input_ids)
frozen_inputs_embeds = torch.cat(
    [frozen_prefix, frozen_text_embeddings],
    dim=1,
)

frozen_outputs = frozen_gpt2(
    inputs_embeds=frozen_inputs_embeds,
    attention_mask=extended_attention_mask,
    labels=extended_labels,
)
frozen_outputs.loss.backward()

assert frozen_prefix.grad is not None
assert frozen_prefix.grad.abs().sum() > 0
assert all(parameter.grad is None for parameter in frozen_gpt2.parameters())

print("PASS: frozen GPT-2 still propagates gradient to visual prefix")

PASS: frozen GPT-2 still propagates gradient to visual prefix


## 11. Kiểm tra nhiều prefix length và batch size

Production code không được giả định `P = 10` hoặc một batch size cố định. Các kiểm tra sau xác nhận hàm hoạt động với nhiều cấu hình.

In [12]:
for batch_size in (1, 4):
    for prefix_length in (1, 5, 10):
        test_input_ids = torch.arange(
            1,
            batch_size * 6 + 1,
            dtype=torch.long,
        ).reshape(batch_size, 6)
        test_attention_mask = torch.ones_like(test_input_ids)
        test_attention_mask[:, -2:] = 0

        result = build_extended_labels(
            test_input_ids,
            test_attention_mask,
            prefix_length,
        )

        assert result.shape == (
            batch_size,
            prefix_length + 6,
        )
        assert torch.all(result[:, :prefix_length] == -100)
        assert torch.all(result[:, -2:] == -100)

print("PASS: dynamic batch size and prefix length")

PASS: dynamic batch size and prefix length


## 12. Các lỗi đầu vào cần được phát hiện sớm

Một hàm tensor nhỏ vẫn nên báo lỗi rõ ràng khi:

- `input_ids` không có shape `[B, L]`.
- `attention_mask` không cùng shape với `input_ids`.
- `input_ids` không dùng `torch.long`.
- `prefix_length` không phải số nguyên dương.

In [13]:
def expect_error(error_type, function, *args):
    try:
        function(*args)
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")


expect_error(
    ValueError,
    build_extended_labels,
    input_ids[0],
    attention_mask[0],
    P,
)
expect_error(
    ValueError,
    build_extended_labels,
    input_ids,
    attention_mask[:, :-1],
    P,
)
expect_error(
    TypeError,
    build_extended_labels,
    input_ids.float(),
    attention_mask,
    P,
)
expect_error(
    ValueError,
    build_extended_labels,
    input_ids,
    attention_mask,
    0,
)

print("PASS: invalid inputs are rejected with clear errors")

PASS: invalid inputs are rejected with clear errors


## 13. Các lỗi thiết kế thường gặp

### Dùng `0` thay cho `-100` ở prefix labels

Token ID `0` có thể là token hợp lệ. GPT-2 sẽ bị yêu cầu dự đoán token đó tại prefix và tạo mục tiêu huấn luyện sai.

### Chỉ mở rộng embeddings, không mở rộng labels

`inputs_embeds` và labels phải có cùng sequence length `P + L`.

### Quên đổi padding labels thành `-100`

`attention_mask = 0` không nên được xem là thay thế cho `labels = -100` trong loss. Hãy bỏ qua padding một cách tường minh trong labels.

### Tự shift labels trước khi gọi GPT-2

Hugging Face GPT-2 tự shift khi nhận `labels`. Shift thủ công thêm một lần sẽ làm mục tiêu lệch vị trí.

### Tạo prefix labels trên CPU khi model ở GPU

Tensor mới phải dùng `device=input_ids.device`; nếu không, phép concatenate sẽ lỗi.

### Sửa `input_ids` tại chỗ

Luôn clone trước khi gán `-100`, vì `input_ids` còn cần cho GPT-2 token embedding.

### Dùng `torch.no_grad()` quanh GPT-2 khi train Mapper

Freeze tham số GPT-2 bằng `requires_grad=False`. Không dùng `torch.no_grad()` nếu cần gradient truyền về Mapper.

## 14. Interface đề xuất khi đưa vào `ClipCaptionModel`

Notebook chưa sửa source code. Sau khi nhóm thống nhất, logic có thể được đặt trong `forward()` của model theo dạng:

```python
def forward(
    self,
    image_embed,
    input_ids,
    attention_mask,
    labels=None,
):
    prefix_embeddings = self.mapper(image_embed)
    prefix_length = prefix_embeddings.size(1)

    text_embeddings = self.gpt2.get_input_embeddings()(input_ids)
    inputs_embeds = torch.cat(
        [prefix_embeddings, text_embeddings],
        dim=1,
    )

    extended_attention_mask = extend_attention_mask(
        attention_mask,
        prefix_length,
    )

    extended_labels = None
    if labels is not None:
        extended_labels = build_extended_labels(
            labels,
            attention_mask,
            prefix_length,
        )

    return self.gpt2(
        inputs_embeds=inputs_embeds,
        attention_mask=extended_attention_mask,
        labels=extended_labels,
    )
```

Nên lấy `prefix_length` từ `prefix_embeddings.size(1)` thay vì hard-code để output thực tế của Mapper luôn đồng bộ với labels.

## 15. Training và generation

### Training hoặc validation loss

Truyền `labels=batch["input_ids"]`. Model tạo `extended_labels` và GPT-2 trả về `outputs.loss`.

### Generation

Không cần labels vì model không dùng teacher-forcing loss. Visual prefix và attention mask vẫn cần thiết. Generation là một luồng riêng vì sequence và attention mask tăng dần khi model sinh token mới.

## 16. Checklist

- [ ] Input là `input_ids [B, L]`, `attention_mask [B, L]` và `prefix_length`.
- [ ] Output là `[B, P + L]`.
- [ ] `P` vị trí đầu đều bằng `-100`.
- [ ] Padding caption đều bằng `-100`.
- [ ] Caption token hợp lệ giữ nguyên token ID.
- [ ] Labels có dtype `torch.long`.
- [ ] Tensor mới nằm cùng device với `input_ids`.
- [ ] Không sửa `input_ids` tại chỗ.
- [ ] Không hard-code prefix length hoặc batch size.
- [ ] Không shift labels thủ công.
- [ ] GPT-2 trả về loss hữu hạn.
- [ ] Loss truyền gradient về visual prefix và Mapper.
- [ ] Dataset không trả labels; model tạo labels khi tham số `labels` khác `None`.